# ColonySearch — Strategy Comparison

Head-to-head benchmark of three search strategies on the same 10-node graph.

| Strategy | Routing | Warm-up | Expected nodes/query |
|----------|---------|---------|----------------------|
| **Broadcast** | None — all 10 nodes queried directly | None | 10 (always) |
| **Random Routing** | Uniform-weight neighbor sampling (ACO_ALPHA=0, ACO_BETA=0) | None | < 10 |
| **ACO Routing** | Pheromone + reputation weighted sampling | 50 round-robin warm-up queries | < 10 |

**Same graph for all three** — one `Network` built once with `seed=42`, pheromone and
reputation tables reset to defaults before each strategy.

Metric: **NDCG@3**, Precision@3, Recall@3.  
Secondary metric: average number of nodes queried per query (efficiency).

In [ ]:
import sys
import contextlib
import io
import json
import random
import time
from copy import deepcopy
from math import log2
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd

# Walk up from the notebook dir to find the project root (contains data/dbs/)
_ROOT = Path().resolve()
for _ in range(6):
    if (_ROOT / 'data' / 'dbs').exists():
        break
    _ROOT = _ROOT.parent
sys.path.insert(0, str(_ROOT))

import search as search_module
import node as node_mod
from network import Network, INITIAL_PHEROMONE
from data.tuning.ground_truth import generate_llm_doc_queries, load_shared_ground_truth

DATA_DIR = _ROOT / 'data'
print(f'Project root : {_ROOT}')
print(f'Data dir     : {DATA_DIR}')

## Experiment Configuration

Edit this cell to swap in the PSO-tuned parameters from `network_benchmark.ipynb`.

In [ ]:
# ── Evaluation budget ─────────────────────────────────────────────────────────
K                = 3    # NDCG / Precision / Recall cut-off
N_WARMUP         = 50   # round-robin warm-up queries for ACO (Strategy 3 only)
N_MEASUREMENT    = 30   # queries used to compute metrics for all strategies
SAMPLE_SEED      = 42   # reproducible query sub-sampling

# ── Tuned ACO parameters (Strategy 3) ────────────────────────────────────────
# Replace these with the best_params from network_benchmark.ipynb if available.
ACO_REP_WEIGHT     = node_mod.REP_WEIGHT      # 0.2
ACO_REP_DECAY      = node_mod.REP_DECAY       # 0.15
ACO_PHEROMONE_EVAP = node_mod.PHEROMONE_EVAP  # 0.1
ACO_ALPHA          = node_mod.ACO_ALPHA       # 1.0
ACO_BETA           = node_mod.ACO_BETA        # 1.0
ACO_NETWORK_TOP_K  = node_mod.NETWORK_TOP_K   # 10
ACO_DEFAULT_TTL    = node_mod.DEFAULT_TTL     # 3

# ── Graph topology (shared across all three strategies) ───────────────────────
EDGE_PROBABILITY = 0.4   # same as the production default in node.py
GRAPH_SEED       = 42

# ── Node directories (absolute so SQLite finds the real DBs) ─────────────────
NODE_IDS = [str(_ROOT / f'data/dbs/Node_{i}') for i in range(1, 11)]

print(f'Nodes         : {len(NODE_IDS)}')
print(f'K             : {K}')
print(f'Warm-up       : {N_WARMUP}')
print(f'Measurement   : {N_MEASUREMENT}')
print()
print('ACO params:')
print(f'  REP_WEIGHT     = {ACO_REP_WEIGHT}')
print(f'  REP_DECAY      = {ACO_REP_DECAY}')
print(f'  PHEROMONE_EVAP = {ACO_PHEROMONE_EVAP}')
print(f'  ACO_ALPHA      = {ACO_ALPHA}')
print(f'  ACO_BETA       = {ACO_BETA}')
print(f'  NETWORK_TOP_K  = {ACO_NETWORK_TOP_K}')
print(f'  DEFAULT_TTL    = {ACO_DEFAULT_TTL}')

## Ground Truth Queries

Reuses the cached LLM queries from `network_benchmark.ipynb`.  
Relevant URLs are pooled across all nodes (union), so the metric reflects what
the whole network can collectively surface.

In [ ]:
LLM_QUERY_CACHE = DATA_DIR / 'tuning' / 'llm_queries.json'

print('Loading ground truth ...')
queries = generate_llm_doc_queries(
    DATA_DIR,
    n_queries=50,
    cache_path=LLM_QUERY_CACHE,
    seed=None,
    allow_llm_fill=False,
)

per_node_cases = load_shared_ground_truth(DATA_DIR, queries)

# Pool relevant URLs across nodes for each query
pooled: dict[str, set] = {}
for tc in per_node_cases:
    qt = tc['query_text']
    pooled.setdefault(qt, set()).update(tc.get('relevant_urls', set()))

all_test_cases = [
    {'query_text': qt, 'relevant_urls': urls}
    for qt, urls in pooled.items()
    if urls
]

# Sub-sample to N_MEASUREMENT for a fair three-way comparison
rng_sample = random.Random(SAMPLE_SEED)
if len(all_test_cases) > N_MEASUREMENT:
    test_cases = rng_sample.sample(all_test_cases, N_MEASUREMENT)
else:
    test_cases = all_test_cases

# Keep a larger pool for warm-up (Strategy 3 only)
warmup_pool = all_test_cases  # may overlap with test_cases intentionally

print(f'\nTotal unique queries : {len(all_test_cases)}')
print(f'Measurement queries  : {len(test_cases)}')
print(f'Warm-up pool size    : {len(warmup_pool)}')
print('\nPreview:')
for tc in test_cases[:5]:
    print(f"  {tc['query_text'][:90]}  ({len(tc['relevant_urls'])} relevant)")

## Shared Network + Reset Helpers

One `Network` is built with `seed=42` and reused across all three strategies.
Between strategies, `_reset_network()` wipes pheromones and reputation tables
so each strategy starts from identical conditions.

In [ ]:
def _build_shared_network() -> tuple:
    """Build one Network + Node objects for all strategies to share."""
    net = Network.build_random_mesh(
        NODE_IDS, edge_probability=EDGE_PROBABILITY, seed=GRAPH_SEED
    )
    nodes = {}
    for nid in NODE_IDS:
        n = node_mod.Node(nid, net, nid)
        net.attach_node_obj(nid, n)
        nodes[nid] = n
    return net, nodes


def _reset_network(net: Network, nodes: dict) -> None:
    """Wipe all pheromone trails and reputation tables to pristine defaults."""
    for (src, dst) in list(net.graph.edges):
        net.set_pheromone(src, dst, INITIAL_PHEROMONE)
    for n in nodes.values():
        n._rep_table.clear()


def _patch_aco_params(
    rep_weight, rep_decay, phero_evap, alpha, beta, top_k, ttl
) -> None:
    """Patch node module globals — caller must restore them afterwards."""
    node_mod.REP_WEIGHT      = float(rep_weight)
    node_mod.REP_DECAY       = float(rep_decay)
    node_mod.PHEROMONE_EVAP  = float(phero_evap)
    node_mod.ACO_ALPHA       = float(alpha)
    node_mod.ACO_BETA        = float(beta)
    node_mod.NETWORK_TOP_K   = int(top_k)
    node_mod.DEFAULT_TTL     = int(ttl)


# Build once — reused by all three strategies
shared_net, shared_nodes = _build_shared_network()

print(f'Network nodes : {len(list(shared_net.graph.nodes))}')
print(f'Network edges : {len(list(shared_net.graph.edges))} directed edges')
print('Shared network ready.')

## Metric Helpers

In [ ]:
def ndcg_at_k(ranked_urls: list, relevant: set, k: int) -> float:
    if not relevant:
        return 0.0
    actual = sum(
        1.0 / log2(i + 1)
        for i, url in enumerate(ranked_urls[:k], start=1)
        if url in relevant
    )
    ideal_len = min(len(relevant), k)
    ideal = sum(1.0 / log2(i + 1) for i in range(1, ideal_len + 1))
    return actual / ideal if ideal else 0.0


def precision_at_k(ranked_urls: list, relevant: set, k: int) -> float:
    if k == 0:
        return 0.0
    return sum(1 for u in ranked_urls[:k] if u in relevant) / k


def recall_at_k(ranked_urls: list, relevant: set, k: int) -> float:
    if not relevant:
        return 0.0
    return sum(1 for u in ranked_urls[:k] if u in relevant) / len(relevant)


def _mean(lst):
    return float(sum(lst) / len(lst)) if lst else 0.0


print('Metric helpers defined.')

---
## Strategy 1: Broadcast

Call `search.search()` (FTS5 + KNN, no network forwarding) on every node directly.
Merge results by URL, keep highest score per URL, return top-10.

**Nodes queried:** always 10.

In [ ]:
def _broadcast_single(query_text: str, query_vector) -> list:
    """Query all 10 nodes in parallel-ish fashion, deduplicate, return top-10."""
    all_results: list[dict] = []
    for nid in NODE_IDS:
        raw = search_module.search(nid, query_text, query_vector)
        for r in raw:
            d = r.to_dict()
            d['node_id'] = nid
            all_results.append(d)

    # Deduplicate by URL — keep the highest score across all nodes
    seen: dict[str, dict] = {}
    for r in all_results:
        url = r.get('url', '')
        if url not in seen or r['score'] > seen[url]['score']:
            seen[url] = r

    return sorted(seen.values(), key=lambda x: x['score'], reverse=True)[:10]


def evaluate_broadcast(cases: list) -> dict:
    """Run broadcast over all test cases and return aggregate metrics."""
    ndcg_scores, prec_scores, rec_scores, latencies = [], [], [], []
    nodes_queried_per_query = []

    for tc in cases:
        qt  = tc['query_text']
        rel = tc['relevant_urls']

        # Encode query once (lazy model load on first call)
        qv = node_mod._encode_query(qt)

        t0 = time.perf_counter()
        results = _broadcast_single(qt, qv)
        latencies.append(time.perf_counter() - t0)

        ranked_urls = [r.get('url', '') for r in results]
        ndcg_scores.append(ndcg_at_k(ranked_urls, rel, K))
        prec_scores.append(precision_at_k(ranked_urls, rel, K))
        rec_scores.append(recall_at_k(ranked_urls, rel, K))
        nodes_queried_per_query.append(len(NODE_IDS))  # always 10

    return {
        'mean_ndcg':      _mean(ndcg_scores),
        'mean_precision': _mean(prec_scores),
        'mean_recall':    _mean(rec_scores),
        'mean_nodes':     _mean(nodes_queried_per_query),
        'mean_latency_s': _mean(latencies),
        'per_query_ndcg': ndcg_scores,
        'per_query_nodes': nodes_queried_per_query,
    }


print('Strategy 1 (Broadcast) defined.')

---
## Strategy 2: Random Routing

Uses the existing `Node.search()` call stack unchanged.  
Override: set `ACO_ALPHA = 0` and `ACO_BETA = 0` so every neighbor weight
collapses to `pheromone^0 · rep^0 = 1` → uniform random sampling.

Pheromone and reputation tables are reset to defaults before the run.
No warm-up.

**Nodes queried:** counted via unique `node_id` values in returned results.

In [ ]:
def evaluate_random(
    cases: list,
    net: Network,
    nodes: dict,
) -> dict:
    """Run random-routing search over all test cases."""
    # Uniform weights: pheromone^0 = 1, rep^0 = 1 for any value → equal probability
    _patch_aco_params(
        rep_weight=0.2,
        rep_decay=0.15,
        phero_evap=0.1,
        alpha=0.0,  # collapse pheromone signal
        beta=0.0,   # collapse reputation signal
        top_k=ACO_NETWORK_TOP_K,
        ttl=ACO_DEFAULT_TTL,
    )
    _reset_network(net, nodes)

    ndcg_scores, prec_scores, rec_scores, latencies = [], [], [], []
    nodes_queried_per_query = []
    rng = random.Random(SAMPLE_SEED)

    for tc in cases:
        qt  = tc['query_text']
        rel = tc['relevant_urls']

        root = nodes[rng.choice(NODE_IDS)]
        t0 = time.perf_counter()
        with contextlib.redirect_stdout(io.StringIO()):
            results = root.search(qt)
        latencies.append(time.perf_counter() - t0)

        ranked_urls  = [r.get('url', '') for r in results]
        unique_nodes = len(set(r.get('node_id', '') for r in results if r.get('node_id')))

        ndcg_scores.append(ndcg_at_k(ranked_urls, rel, K))
        prec_scores.append(precision_at_k(ranked_urls, rel, K))
        rec_scores.append(recall_at_k(ranked_urls, rel, K))
        nodes_queried_per_query.append(unique_nodes)

    return {
        'mean_ndcg':      _mean(ndcg_scores),
        'mean_precision': _mean(prec_scores),
        'mean_recall':    _mean(rec_scores),
        'mean_nodes':     _mean(nodes_queried_per_query),
        'mean_latency_s': _mean(latencies),
        'per_query_ndcg': ndcg_scores,
        'per_query_nodes': nodes_queried_per_query,
    }


print('Strategy 2 (Random Routing) defined.')

---
## Strategy 3: ACO Routing

Uses the existing `Node.search()` with tuned pheromone / reputation parameters.  
**50 round-robin warm-up queries** seed the pheromone gradients before measurement.

Pheromone and reputation tables are reset to defaults before warm-up so the three
strategies all start from identical state.

**Nodes queried:** counted via unique `node_id` values in returned results.

In [ ]:
def evaluate_aco(
    cases: list,
    net: Network,
    nodes: dict,
    warmup_pool: list,
    n_warmup: int = N_WARMUP,
) -> dict:
    """Run ACO-routed search with warm-up over all test cases."""
    _patch_aco_params(
        rep_weight=ACO_REP_WEIGHT,
        rep_decay=ACO_REP_DECAY,
        phero_evap=ACO_PHEROMONE_EVAP,
        alpha=ACO_ALPHA,
        beta=ACO_BETA,
        top_k=ACO_NETWORK_TOP_K,
        ttl=ACO_DEFAULT_TTL,
    )
    _reset_network(net, nodes)

    # Round-robin warm-up: cycle over every node as root to seed pheromone
    # gradients on all edges before measuring.  Repeat the pool as needed.
    extended_warmup = (warmup_pool * (n_warmup // len(warmup_pool) + 1))[:n_warmup]
    print(f'Running {n_warmup} warm-up queries ...', end='', flush=True)
    for i, wtc in enumerate(extended_warmup):
        root_id = NODE_IDS[i % len(NODE_IDS)]
        with contextlib.redirect_stdout(io.StringIO()):
            nodes[root_id].search(wtc['query_text'])
        if (i + 1) % 10 == 0:
            print(f' {i+1}', end='', flush=True)
    print(' done.')

    # Measurement pass
    ndcg_scores, prec_scores, rec_scores, latencies = [], [], [], []
    nodes_queried_per_query = []
    rng = random.Random(SAMPLE_SEED)

    for tc in cases:
        qt  = tc['query_text']
        rel = tc['relevant_urls']

        root = nodes[rng.choice(NODE_IDS)]
        t0 = time.perf_counter()
        with contextlib.redirect_stdout(io.StringIO()):
            results = root.search(qt)
        latencies.append(time.perf_counter() - t0)

        ranked_urls  = [r.get('url', '') for r in results]
        unique_nodes = len(set(r.get('node_id', '') for r in results if r.get('node_id')))

        ndcg_scores.append(ndcg_at_k(ranked_urls, rel, K))
        prec_scores.append(precision_at_k(ranked_urls, rel, K))
        rec_scores.append(recall_at_k(ranked_urls, rel, K))
        nodes_queried_per_query.append(unique_nodes)

    return {
        'mean_ndcg':      _mean(ndcg_scores),
        'mean_precision': _mean(prec_scores),
        'mean_recall':    _mean(rec_scores),
        'mean_nodes':     _mean(nodes_queried_per_query),
        'mean_latency_s': _mean(latencies),
        'per_query_ndcg': ndcg_scores,
        'per_query_nodes': nodes_queried_per_query,
    }


print('Strategy 3 (ACO Routing) defined.')

---
## Run All Three Strategies

Each strategy resets the shared network to identical starting conditions.

In [ ]:
results = {}

# ── Strategy 1: Broadcast ─────────────────────────────────────────────────────
print('=' * 60)
print('Strategy 1: BROADCAST')
print('=' * 60)
t0 = time.perf_counter()
results['Broadcast'] = evaluate_broadcast(test_cases)
elapsed = time.perf_counter() - t0
print(f"NDCG@{K}: {results['Broadcast']['mean_ndcg']:.4f}  "
      f"Prec@{K}: {results['Broadcast']['mean_precision']:.4f}  "
      f"Recall@{K}: {results['Broadcast']['mean_recall']:.4f}  "
      f"Nodes/q: {results['Broadcast']['mean_nodes']:.1f}  "
      f"({elapsed:.0f}s)")

# ── Strategy 2: Random Routing ────────────────────────────────────────────────
print()
print('=' * 60)
print('Strategy 2: RANDOM ROUTING')
print('=' * 60)
t0 = time.perf_counter()
results['Random'] = evaluate_random(test_cases, shared_net, shared_nodes)
elapsed = time.perf_counter() - t0
print(f"NDCG@{K}: {results['Random']['mean_ndcg']:.4f}  "
      f"Prec@{K}: {results['Random']['mean_precision']:.4f}  "
      f"Recall@{K}: {results['Random']['mean_recall']:.4f}  "
      f"Nodes/q: {results['Random']['mean_nodes']:.1f}  "
      f"({elapsed:.0f}s)")

# ── Strategy 3: ACO Routing ───────────────────────────────────────────────────
print()
print('=' * 60)
print('Strategy 3: ACO ROUTING')
print('=' * 60)
t0 = time.perf_counter()
results['ACO'] = evaluate_aco(test_cases, shared_net, shared_nodes, warmup_pool)
elapsed = time.perf_counter() - t0
print(f"NDCG@{K}: {results['ACO']['mean_ndcg']:.4f}  "
      f"Prec@{K}: {results['ACO']['mean_precision']:.4f}  "
      f"Recall@{K}: {results['ACO']['mean_recall']:.4f}  "
      f"Nodes/q: {results['ACO']['mean_nodes']:.1f}  "
      f"({elapsed:.0f}s)")

print()
print('All strategies complete.')

---
## Results Table

In [ ]:
ndcg_col  = f'NDCG@{K}'
prec_col  = f'Prec@{K}'
rec_col   = f'Recall@{K}'

rows = []
for name, res in results.items():
    rows.append({
        'Strategy':       name,
        ndcg_col:         round(res['mean_ndcg'],      4),
        prec_col:         round(res['mean_precision'],  4),
        rec_col:          round(res['mean_recall'],     4),
        'Nodes / query':  round(res['mean_nodes'],     2),
        'Latency (s)':    round(res['mean_latency_s'], 2),
    })

df = pd.DataFrame(rows).set_index('Strategy')
display(
    df.style
      .highlight_max(subset=[ndcg_col, prec_col, rec_col], color='lightgreen')
      .highlight_min(subset=['Nodes / query', 'Latency (s)'],  color='lightblue')
      .format(precision=4)
)
print(f'\nQueries evaluated: {N_MEASUREMENT}')

---
## Visualisations

In [ ]:
COLORS = {
    'Broadcast': '#4e79a7',
    'Random':    '#f28e2b',
    'ACO':       '#59a14f',
}

strategy_names = list(results.keys())

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# ── Panel 1: Quality metrics bar chart ────────────────────────────────────────
ax = axes[0]
metrics = [ndcg_col, prec_col, rec_col]
x = np.arange(len(metrics))
width = 0.25
for i, name in enumerate(strategy_names):
    vals = [
        results[name]['mean_ndcg'],
        results[name]['mean_precision'],
        results[name]['mean_recall'],
    ]
    ax.bar(x + i * width, vals, width, label=name, color=COLORS[name])

ax.set_xticks(x + width)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score')
ax.set_title('Quality Metrics')
ax.legend()
ax.grid(axis='y', alpha=0.3)

# ── Panel 2: Per-query NDCG box plot ─────────────────────────────────────────
ax = axes[1]
data   = [results[n]['per_query_ndcg'] for n in strategy_names]
colors = [COLORS[n] for n in strategy_names]
bp = ax.boxplot(data, labels=strategy_names, patch_artist=True, notch=False)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_ylim(-0.05, 1.05)
ax.set_ylabel(f'NDCG@{K}')
ax.set_title(f'Per-query NDCG@{K} Distribution')
ax.grid(axis='y', alpha=0.3)

# ── Panel 3: Average nodes queried bar chart ──────────────────────────────────
ax = axes[2]
node_counts = [results[n]['mean_nodes'] for n in strategy_names]
bar_colors  = [COLORS[n] for n in strategy_names]
bars = ax.bar(strategy_names, node_counts, color=bar_colors)
ax.set_ylim(0, 11)
ax.set_ylabel('Avg nodes queried')
ax.set_title('Network Load per Query')
ax.axhline(10, linestyle='--', color='gray', linewidth=1, label='Max (10)')
ax.legend()
ax.grid(axis='y', alpha=0.3)
for bar, val in zip(bars, node_counts):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.15,
        f'{val:.1f}',
        ha='center', fontsize=10,
    )

fig.suptitle(
    f'Strategy Comparison — {N_MEASUREMENT} measurement queries, K={K}',
    fontsize=13, y=1.01,
)
fig.tight_layout()
fig.savefig('strategy_comparison_overview.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved → strategy_comparison_overview.png')

In [ ]:
# ── Efficiency scatter: NDCG vs. nodes queried ────────────────────────────────
# Each dot = one query; colour = strategy.
# Upper-left corner is best: high NDCG with few nodes queried.

fig, ax = plt.subplots(figsize=(8, 5))

for name in strategy_names:
    xvals = results[name]['per_query_nodes']
    yvals = results[name]['per_query_ndcg']
    ax.scatter(
        xvals, yvals,
        color=COLORS[name], label=name,
        alpha=0.6, edgecolors='white', linewidths=0.5, s=60,
    )
    # Mean marker
    ax.scatter(
        [_mean(xvals)], [_mean(yvals)],
        color=COLORS[name], marker='D', s=120,
        edgecolors='black', linewidths=1.2, zorder=5,
    )

ax.set_xlabel('Nodes queried (unique node_ids in results)')
ax.set_ylabel(f'NDCG@{K}')
ax.set_title(f'Efficiency: NDCG@{K} vs. Nodes Queried\n(diamonds = per-strategy mean)')
ax.set_xlim(0, 11)
ax.set_ylim(-0.05, 1.05)
ax.legend()
ax.grid(alpha=0.3)

fig.tight_layout()
fig.savefig('strategy_comparison_efficiency.png', dpi=120)
plt.show()
print('Saved → strategy_comparison_efficiency.png')

In [ ]:
# ── Per-query NDCG line plot — shows where each strategy wins/loses ───────────

fig, ax = plt.subplots(figsize=(12, 4))

x_idx = np.arange(N_MEASUREMENT)
for name in strategy_names:
    ax.plot(
        x_idx, results[name]['per_query_ndcg'],
        label=name, color=COLORS[name], alpha=0.8, linewidth=1.5,
    )

ax.set_xlabel('Query index')
ax.set_ylabel(f'NDCG@{K}')
ax.set_title(f'Per-query NDCG@{K} across {N_MEASUREMENT} measurement queries')
ax.set_ylim(-0.05, 1.05)
ax.legend()
ax.grid(alpha=0.3)

fig.tight_layout()
fig.savefig('strategy_comparison_perquery.png', dpi=120)
plt.show()
print('Saved → strategy_comparison_perquery.png')

In [ ]:
# ── Summary print ─────────────────────────────────────────────────────────────
print('\n' + '=' * 60)
print('SUMMARY')
print('=' * 60)
print(df.to_string())
print()

best_ndcg  = max(strategy_names, key=lambda n: results[n]['mean_ndcg'])
fewest_nodes = min(strategy_names, key=lambda n: results[n]['mean_nodes'])

aco_ndcg   = results['ACO']['mean_ndcg']
bcast_ndcg = results['Broadcast']['mean_ndcg']
rand_ndcg  = results['Random']['mean_ndcg']

print(f'Best NDCG@{K}        : {best_ndcg} ({results[best_ndcg]["mean_ndcg"]:.4f})')
print(f'Fewest nodes/query : {fewest_nodes} ({results[fewest_nodes]["mean_nodes"]:.1f})')
print()
print(f'ACO vs. Broadcast  : {(aco_ndcg - bcast_ndcg):+.4f} NDCG@{K}')
print(f'ACO vs. Random     : {(aco_ndcg - rand_ndcg):+.4f} NDCG@{K}')
print(f'ACO node savings   : {results["Broadcast"]["mean_nodes"] - results["ACO"]["mean_nodes"]:.1f} '
      f'nodes/query vs. Broadcast')